In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup, Comment
import io

# Step 1: Data Acquisition

In [41]:
def scrape_mvp_voting(season: int = 2025) -> pd.DataFrame:
    """
    Scrape MVP Award Voting table from Basketball Reference.
    
    Args:
        season: Year of the award (e.g., 2000 for the 1999-00 season)
    """
    url = f"https://www.basketball-reference.com/awards/awards_{season}.html"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, "html.parser")
    
    # The MVP table usually has id="mvp"
    # It might be commented out to save bandwidth, so we check both
    table_html = None
    table = soup.find("table", {"id": "mvp"})
    
    if table:
        table_html = str(table)
    else:
        # Check inside comments
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            if 'id="mvp"' in comment:
                table_html = comment
                break
                
    if not table_html:
        raise ValueError(f"Could not find MVP table for season {season}")

    # Parse with Pandas
    # header=1 tells pandas to use the second row as the main header 
    # (ignoring the top 'Voting', 'Per Game' grouping row)
    df = pd.read_html(io.StringIO(table_html), header=1)[0]
    
    # Clean up columns
    # Sometimes read_html with header=1 leaves "Unnamed" columns or duplicates
    # We explicitly rename key columns to be safe
    
    # 1. Remove repeated header rows (scraped data often has headers repeated every 20 rows)
    df = df[df['Player'] != 'Player']
    
    # 2. Handle the "Team" column issue
    # The column is usually "Tm", but might have spaces. 
    # We rename columns to be standard.
    df.columns = [c.strip() for c in df.columns]
    
    # 3. Convert Numeric Columns
    # We leave 'Player', 'Tm', 'Pos' as strings, convert rest to numeric
    text_cols = ['Player', 'Tm', 'Pos', 'Rank']
    for col in df.columns:
        if col not in text_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill NaN in 'First' (First Place Votes) with 0, as empty usually means 0 votes
    if 'First' in df.columns:
        df['First'] = df['First'].fillna(0)

    # Add Season column
    start_year = season - 1
    season_str = f"{start_year}-{season % 100:02d}"    
    df.insert(0, 'Season', season_str)
            
    return df.reset_index(drop=True)

In [42]:
df = scrape_mvp_voting(2025)
df

,Season,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,...,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
0,2024-25,1,Shai Gilgeous-Alexander,26,OKC,71,913,1000,0.913,76,...,32.7,5.0,6.4,1.7,1.0,0.519,0.375,0.898,16.7,0.309
1,2024-25,2,Nikola Jokić,29,DEN,29,787,1000,0.787,70,...,29.6,12.7,10.2,1.8,0.6,0.576,0.417,0.800,16.4,0.307
2,2024-25,3,Giannis Antetokounmpo,30,MIL,0,470,1000,0.470,67,...,30.4,11.9,6.5,0.9,1.2,0.601,0.222,0.617,11.5,0.241
3,2024-25,4,Jayson Tatum,26,BOS,0,311,1000,0.311,72,...,26.8,8.7,6.0,1.1,0.5,0.452,0.343,0.814,9.5,0.174
4,2024-25,5,Donovan Mitchell,28,CLE,0,74,1000,0.074,71,...,24.0,4.5,5.0,1.3,0.2,0.443,0.368,0.823,7.6,0.163
5,2024-25,6,LeBron James,40,LAL,0,16,1000,0.016,70,...,24.4,7.8,8.2,1.0,0.6,0.513,0.376,0.782,7.7,0.152
6,2024-25,7T,Cade Cunningham,23,DET,0,12,1000,0.012,70,...,26.1,6.1,9.1,1.0,0.8,0.469,0.356,0.846,5.9,0.115
7,2024-25,7T,Anthony Edwards,23,MIN,0,12,1000,0.012,79,...,27.6,5.7,4.5,1.2,0.6,0.447,0.395,0.837,8.4,0.140
8,2024-25,9,Stephen Curry,36,GSW,0,2,1000,0.002,70,...,24.5,4.4,6.0,1.1,0.4,0.448,0.397,0.933,7.9,0.168
9,2024-25,10T,Jalen Brunson,28,NYK,0,1,1000,0.001,65,...,26.0,2.9,7.3,0.9,0.1,0.488,0.383,0.821,8.3,0.172


In [43]:
def scrape_nba_standings(season: int = 2025) -> pd.DataFrame:
    """
    Scrape NBA Standings. 
    - Prioritizes "Conference" standings (modern).
    - Falls back to "Division" standings for older seasons (pre-1971).
    - Cleans out non-team rows (division headers).
    """
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}_standings.html"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Define sets of IDs to try in order of preference
    # Set 1: Modern Conference Standings (Preferred because they are cleaner)
    # Set 2: Division Standings (Fallback for pre-1971 or if Conf not found)
    id_sets = [
        [("confs_standings_E", "East"), ("confs_standings_W", "West")],
        [("divs_standings_E", "East"), ("divs_standings_W", "West")]
    ]
    
    dfs = []
    
    # Try the ID sets one by one
    for id_set in id_sets:
        current_set_dfs = []
        
        for table_id, conf_name in id_set:
            # --- 1. Locate Table ---
            table_html = None
            table = soup.find("table", {"id": table_id})
            
            if table:
                table_html = str(table)
            else:
                # Check comments if not found in main DOM
                comments = soup.find_all(string=lambda text: isinstance(text, Comment))
                for comment in comments:
                    if f'id="{table_id}"' in comment:
                        table_html = comment
                        break
            
            if not table_html:
                # If one table in the pair is missing, this set is invalid
                break 

            # --- 2. Parse Table ---
            try:
                df = pd.read_html(io.StringIO(table_html))[0]
            except ValueError:
                break

            # Rename first column to 'Team'
            df.rename(columns={df.columns[0]: "Team"}, inplace=True)
            
            # --- 3. Clean Data ---
            # Remove rows where 'W' is not a number (removes Division headers)
            if "W" in df.columns:
                df = df[df["W"].astype(str).str.match(r'^\d+$', na=False)]
            
            # Clean Team Name (remove * for playoff teams)
            df["Team"] = df["Team"].str.replace("*", "", regex=False)
            
            # Add Conference Column
            df["Conference"] = conf_name
            
            current_set_dfs.append(df)
        
        # If we successfully found both tables (East and West) for this set, stop looking
        if len(current_set_dfs) == 2:
            dfs = current_set_dfs
            break
    
    if not dfs:
        # If we still have nothing, print the URL to help debug
        raise ValueError(f"Could not find standings tables for season {season}. URL: {url}")
        
    # Concatenate vertically
    final_df = pd.concat(dfs, ignore_index=True)
    
    # Convert numeric columns
    cols_to_keep_text = ['Team', 'Conference']
    for col in final_df.columns:
        if col not in cols_to_keep_text:
            final_df[col] = pd.to_numeric(final_df[col], errors='coerce')
            
    return final_df

In [45]:
mvp_dfs = []
standing_dfs = []

for season in range(2010, 2026):
    mvp_df = scrape_mvp_voting(season)
    mvp_dfs.append(mvp_df)

    standing_df = scrape_nba_standings(season)
    standing_dfs.append(standing_df)


all_mvp_dfs = pd.concat(mvp_dfs, ignore_index=True)
all_mvp_dfs.to_csv("data/nba_mvp_voting_2010_to_2025.csv", index=False)

all_standing_dfs = pd.concat(standing_dfs, ignore_index=True)
all_standing_dfs.to_csv("data/nba_standings_2010_to_2025.csv", index=False)

HTTPError: 429 Client Error: Too Many Requests for url: https://www.basketball-reference.com/awards/awards_2010.html

In [3]:
def scrape_per_game_stats(season: int = 2026) -> pd.DataFrame:
   """
   Scrape per-game stats from Basketball Reference.
  
   Args:
       season: NBA season year (e.g., 2025 for 2024-25 season)
  
   Returns:
       DataFrame with player per-game stats
   """
   url = f"https://www.basketball-reference.com/leagues/NBA_{season}_per_game.html"
  
   headers = {
       "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
       "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
       "Accept-Language": "en-US,en;q=0.5",
       "Referer": "https://www.basketball-reference.com/",
   }
  
   print(f"Fetching {url}...")
   response = requests.get(url, headers=headers, timeout=30)
   response.raise_for_status()
  
   soup = BeautifulSoup(response.content, "html.parser")
  
   # Find the per_game table (may be in comments)
   table = soup.find("table", {"id": "per_game_stats"})
  
   if table is None:
       comments = soup.find_all(string=lambda text: isinstance(text, Comment))
       for comment in comments:
           if "per_game_stats" in comment:
               comment_soup = BeautifulSoup(comment, "html.parser")
               table = comment_soup.find("table", {"id": "per_game_stats"})
               if table:
                   break
  
   if table is None:
       raise ValueError("Could not find per_game_stats table")
  
   df = pd.read_html(io.StringIO(str(table)))[0]
  
   # Clean up the dataframe
   # 1. Remove repeating header rows (where Rk is 'Rk')
   df = df[df['Rk'] != 'Rk']
  
   # 2. Handle Numeric Conversion safely
   # We infer objects first, then force numeric on everything except the known string columns
   df = df.infer_objects()
  
   cols_to_ignore = ['Player', 'Pos', 'Team', 'Awards']
   for col in df.columns:
       if col not in cols_to_ignore:
           df[col] = pd.to_numeric(df[col], errors='coerce')


   return df

In [4]:
df = scrape_per_game_stats(2026)
df.to_csv("data/nba_per_game_stats_2025_to_2026.csv", index=False)

Fetching https://www.basketball-reference.com/leagues/NBA_2026_per_game.html...
